#  Energy Consumption Modeling for UAVs Based on Flight Data and Mission Parameters :
## 1. Data analysis and feature extraction

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.ndimage import gaussian_filter1d

In [2]:
from utils import clean_flight_df, plot_flight_altitude_speed, MAE, RMSE, MAPE

### M100 Data

In [3]:
df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)
cols = df.columns.tolist()

C:\Users\raman\AppData\Local\Temp\ipykernel_7820\2046151961.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)


### Data Cleaning

Sanity check

In [4]:
clean_df = clean_flight_df(df)

c:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\utils.py:251: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_day'] = pd.to_datetime(df['time_day'], errors='coerce').dt.time


### Find different phase of flight using  altitude, horizontal and vertical speed. 

In [6]:
df = df.sort_values(["flight", "time"]).reset_index(drop=True)
df["altitude_measured"] = df["position_z"] - df.groupby("flight")["position_z"].transform("first")
df["horizontal_speed"] = np.sqrt(df["velocity_x"]**2 + df["velocity_y"]**2)
df['alt_diff'] = df.groupby('flight')['altitude_measured'].diff()
df['max_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('max')
df['min_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('min')

df["dt"] = df.groupby("flight")["time"].diff()

df["vz_from_alt_raw"] = df["alt_diff"] / df["dt"]

# Moving average smoothing
# df["vz_from_alt"] = (
#     df.groupby("flight")["vz_from_alt_raw"]
#       .transform(lambda x: x.rolling(window=10, center=True, min_periods=1).mean())
# )

# Gaussian filter smoothing
df["vz_from_alt"] = (
    df.groupby("flight")["vz_from_alt_raw"]
      .transform(lambda x: gaussian_filter1d(x, sigma=5, mode="nearest"))
)

df["power_w"] = df["battery_voltage"] * df["battery_current"]

### Velocity from altitude diff and velocity_z are not same e.g. in 120
So lets do a sanity check.

In [7]:
# Select flights to inspect
flights_to_plot = [120, 79, 279]   # add more flight numbers here if you like
sub = df[df["flight"].isin(flights_to_plot)].copy()

# Prepare data for Plotly (melt to long format for nice legends)
plot_df = sub.melt(
    id_vars=["flight", "time"],
    value_vars=["velocity_z", "vz_from_alt"],
    var_name="source",
    value_name="vz"
)

# Plot both velocities over time, faceted by Flight
fig = px.line(
    plot_df,
    x="time",
    y="vz",
    color="source",
    facet_row="flight",        # or facet_col="flight" if you prefer columns
    title="Reported vs Altitude-derived Vertical Velocity",
    labels={
        "time": "Time",
        "vz": "Vertical velocity (m/s)",
        "source": "Signal source"
    }
)

fig.update_layout(height=300*len(flights_to_plot))  # adjust height for number of flights
fig.show()

#### Definitions of various phases

In [8]:

# Thresholds
NEARLY_ZERO_HORIZONTAL_VEL   = 0.3   # m/s 
NEARLY_ZERO_VERTICAL_VEL     = 0.2   # m/s

CRUISE_MAX_VERTICAL_VEL    = 0.6   # m/s
CRUISE_MIN_VERTICAL_VEL    = -0.6  # m/s

TAXI_ALTITUDE_MAX          = 1.0   # m   # max altitude to be considered taxiing
NEARLY_ZERO_POWER_W        = 5.0  # W   # max power to be considered taxiing

# Start with a default phase
df['phase'] = 'other'

# --- Taxi: low altitude, nearly no motion ---
taxi_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) &
    (df['power_w']      < NEARLY_ZERO_POWER_W)
)
df.loc[(df['phase'] == 'other') & taxi_mask, 'phase'] = 'taxi'


# --- Hover: nearly no motion ---
# 1. horizontal speed below hover max
# 2. vertical speed below hover max
hover_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) 
)
df.loc[(df['phase'] == 'other') & hover_mask, 'phase'] = 'hover'

# --- Cruise: nearly no vertical motion, high altitude ---
# 1. vertical speed below cruise max
# 2. altitude above 80% of max altitude in flight
cruise_mask = (
    (df['vz_from_alt'] < CRUISE_MAX_VERTICAL_VEL) &
    (df['vz_from_alt'] > CRUISE_MIN_VERTICAL_VEL) &
    (df['altitude_measured'] > 0.8 * df['max_altitude_flight'])
)
df.loc[(df['phase'] == 'other') & cruise_mask, 'phase'] = 'cruise'

# ---- CLIMB ----
# 1. vertical speed is greater than climb vertical speed minimum
climb_mask = (df['vz_from_alt'] > CRUISE_MAX_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & climb_mask, 'phase'] = 'climb'

# ---- DESCENT ----
# 1. vertical speed is less than negative of climb vertical speed minimum
descent_mask = (df['vz_from_alt'] < CRUISE_MIN_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & descent_mask, 'phase'] = 'descent'




### Visualize different phase of flight with Line graph

In [9]:
flight_id = 79
d = df[df["flight"] == flight_id].copy()

# ------------------ Define colors for phases ------------------
phase_colors = {
    "climb":   "rgba(0,255,0,0.25)",      # light green
    "descent": "rgba(255,0,0,0.25)",      # light red
    "hover":   "rgba(255,165,0,0.25)",    # light orange
    "cruise":  "rgba(0,0,255,0.25)",      # light blue
    "taxi":    "rgba(255,255,0,0.25)",    # light yellow
    "other":   "rgba(150,150,150,0.15)"   # light gray
}

# ------------------ Build customdata for hover ------------------
customdata = np.stack((
    d["altitude_measured"],
    d["vz_from_alt"],
    d["horizontal_speed"],
    d["phase"]
), axis=-1)

fig = plot_flight_altitude_speed(
    d,
    customdata=customdata,
    phase_colors=phase_colors,
    flight_id=flight_id,
    show=True
)

### Task two

In [11]:
def compute_wind_vector(df):
    rad = np.deg2rad(df["wind_angle"])
    wind_x = -df["wind_speed"] * np.cos(rad)
    wind_y = -df["wind_speed"] * np.sin(rad)
    return wind_x, wind_y

def compute_airspeed(df, wind_x, wind_y):
    return np.sqrt(
        (df["velocity_x"] - wind_x)**2 +
        (df["velocity_y"] - wind_y)**2
    )

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [13]:
P_real     = df["power_w"].values

# wind speed calculations
wind_x, wind_y = compute_wind_vector(df)
df["wind_x"] = wind_x
df["wind_y"] = wind_y
df["airspeed"] = compute_airspeed(df, wind_x, wind_y)

V_air      = df["airspeed"].values
Vz         = df["vz_from_alt"].values
segment    = df["phase"].values
payload_kg = df["payload"].values / 1000.0

# ------------------------------------------------------
# 2. Physical Parameters for DJI M100
# ------------------------------------------------------
g   = 9.81
rho = 1.225

M0   = 3.68                      # empty mass [kg]
mass = M0 + payload_kg           # total mass [kg]
W    = mass * g                  # weight [N]

# Rotor geometry (13-inch props)
R_prop = 0.165                   # rotor radius [m]
A_rot  = np.pi * R_prop**2       # single rotor disk area [m^2]
A_tot  = 4 * A_rot               # total rotor disk area [m^2]

# Reference area & drag coefficient (from literature)
RAD   = 0.43                     # rotor axis distance [m]
S_ref = np.pi * (RAD / 2.0)**2   # reference area [m^2]
Cd    = 0.316                    # from external UAS paper

seg = segment  # shorthand

# ------------------------------------------------------
# 3. Induced Velocity & Hover Power (Momentum Theory)
# ------------------------------------------------------
vh = np.sqrt(W / (2.0 * rho * A_tot))   # induced velocity [m/s]

P_hover_base = (W ** 1.5) / np.sqrt(2.0 * rho * A_tot)  # hover induced power

# ------------------------------------------------------
# 4. Compute Base Mechanical Power (Segment-wise)
# ------------------------------------------------------
P_mech_base = np.zeros_like(P_real, dtype=float)

# --- Hover ---
mask_hover = (seg == "hover")
P_mech_base[mask_hover] = P_hover_base[mask_hover]

# --- Climb ---
mask_climb = (seg == "climb")
if mask_climb.any():
    RoC   = Vz[mask_climb]
    vh_c  = vh[mask_climb]
    ratio = RoC / (2.0 * vh_c)
    P_mech_base[mask_climb] = P_hover_base[mask_climb] * (
        ratio + np.sqrt(ratio**2 + 1.0)
    )

# --- Descent ---
mask_descent = (seg == "descent")
if mask_descent.any():
    RoD   = Vz[mask_descent]
    vh_d  = vh[mask_descent]
    ratio_d = RoD / (2.0 * vh_d)
    ratio_d = np.clip(ratio_d, -5.0, 5.0)
    P_desc = P_hover_base[mask_descent] * (
        ratio_d + np.sqrt(ratio_d**2 + 1.0)
    )
    P_mech_base[mask_descent] = np.maximum(P_desc, 0.3 * P_hover_base[mask_descent])

# --- Cruise ---
mask_cruise = (seg == "cruise")
if mask_cruise.any():
    Vc = V_air[mask_cruise]
    P_parasite = 0.5 * rho * Cd * S_ref * (Vc**3)
    P_mech_base[mask_cruise] = P_hover_base[mask_cruise] + P_parasite

# Transition = ignored
mask_transition = (seg == "other")


# ------------------------------------------------------
# 5. Estimate Effective Efficiency η per Segment
# ------------------------------------------------------
eta_seg = {}

for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_real > 30) & (P_mech_base > 0)
    if mask_s.sum() < 50:
        continue

    Pm = P_mech_base[mask_s]
    Pr = P_real[mask_s]

    # eta = Σ Pm² / Σ (Pm * Pr)
    eta_hat = np.sum(Pm**2) / np.sum(Pm * Pr)
    eta_seg[s] = eta_hat

print("Estimated segment-wise efficiencies (mechanical → battery):")
for s, eta_hat in eta_seg.items():
    print(f"  {s:8s}: eta_seg = {eta_hat:.3f}")


# ------------------------------------------------------
# 6. Build Final Battery Power Model
# ------------------------------------------------------
P_model_phys = np.full_like(P_real, np.nan, dtype=float)

for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_mech_base > 0)
    if s not in eta_seg:
        continue
    P_model_phys[mask_s] = P_mech_base[mask_s] / eta_seg[s]

df["P_model_phys"] = P_model_phys


# ------------------------------------------------------
# 7. Evaluate Power Model Accuracy
# ------------------------------------------------------
valid_eval = (P_real > 30) & (seg != "transition") & ~np.isnan(P_model_phys)

MAE  = mean_absolute_error(P_real[valid_eval], P_model_phys[valid_eval])
RMSE = np.sqrt(mean_squared_error(P_real[valid_eval], P_model_phys[valid_eval]))
MAPE = (np.abs(P_model_phys[valid_eval] - P_real[valid_eval]) /
        P_real[valid_eval]).mean() * 100.0

print("\n===== FINAL PHYSICS MODEL RESULTS =====")
print(f"MAE  = {MAE:.2f} W")
print(f"RMSE = {RMSE:.2f} W")
print(f"MAPE = {MAPE:.2f} %")


print("\n===== ERROR BY SEGMENT (NO TRANSITION) =====")
for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = (seg == s) & (P_real > 30) & ~np.isnan(P_model_phys)
    if mask_s.sum() < 50:
        continue
    mae_s  = mean_absolute_error(P_real[mask_s], P_model_phys[mask_s])
    rmse_s = np.sqrt(mean_squared_error(P_real[mask_s], P_model_phys[mask_s]))
    print(f"{s:10s} | N={mask_s.sum():6d} | MAE={mae_s:8.2f} | RMSE={rmse_s:8.2f}")


# ------------------------------------------------------
# 8. Save Output
# ------------------------------------------------------
out_name = "M100_task2_physics_segment_eff.csv"
df.to_csv(out_name, index=False)
print(f"\nSaved: {out_name}")


Estimated segment-wise efficiencies (mechanical → battery):
  hover   : eta_seg = 0.829
  climb   : eta_seg = 0.534
  descent : eta_seg = 0.462
  cruise  : eta_seg = 0.611

===== FINAL PHYSICS MODEL RESULTS =====
MAE  = 84.06 W
RMSE = 118.01 W
MAPE = 28.93 %

===== ERROR BY SEGMENT (NO TRANSITION) =====
hover      | N= 12474 | MAE=  201.86 | RMSE=  211.64
climb      | N= 29949 | MAE=   57.04 | RMSE=   82.64
descent    | N= 48264 | MAE=   45.77 | RMSE=   64.36
cruise     | N=103112 | MAE=   95.58 | RMSE=  129.75

Saved: M100_task2_physics_segment_eff.csv


### Task 3

In [22]:
allowed_phases = ["cruise", "descent", "climb", "hover"]

df["phase"] = df["phase"].str.lower().str.strip()   # normalize

df_filtered = df[df["phase"].isin(allowed_phases)]

In [23]:
import math
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


# ========= 1. Quaternion → roll, pitch, yaw =========
# Quaternion assumed in (x, y, z, w) convention

def quat_to_euler(x, y, z, w):
    # Normalize to avoid numerical issues
    norm = math.sqrt(x*x + y*y + z*z + w*w)
    if norm == 0:
        return 0.0, 0.0, 0.0
    x /= norm
    y /= norm
    z /= norm
    w /= norm

    # roll (x-axis rotation)
    sinr_cosp = 2.0 * (w * x + y * z)
    cosr_cosp = 1.0 - 2.0 * (x * x + y * y)
    roll = math.atan2(sinr_cosp, cosr_cosp)

    # pitch (y-axis rotation)
    sinp = 2.0 * (w * y - z * x)
    if abs(sinp) >= 1:
        # use 90 degrees if out of range
        pitch = math.copysign(math.pi / 2.0, sinp)
    else:
        pitch = math.asin(sinp)

    # yaw (z-axis rotation)
    siny_cosp = 2.0 * (w * z + x * y)
    cosy_cosp = 1.0 - 2.0 * (y * y + z * z)
    yaw = math.atan2(siny_cosp, cosy_cosp)

    return roll, pitch, yaw

# Apply to the whole dataframe
rpy = df_filtered.apply(
    lambda row: pd.Series(
        quat_to_euler(
            row["orientation_x"],
            row["orientation_y"],
            row["orientation_z"],
            row["orientation_w"],
        ),
        index=["roll", "pitch", "yaw"],
    ),
    axis=1,
)

df_filtered = pd.concat([df_filtered, rpy], axis=1)

In [26]:
df_filtered["phase"] = df_filtered["phase"].str.lower().str.strip()

df_filtered["is_cruise"]  = df_filtered["phase"] == "cruise"
df_filtered["is_climb"]   = df_filtered["phase"] == "climb"
df_filtered["is_descent"] = df_filtered["phase"] == "descent"
df_filtered["is_hover"]   = df_filtered["phase"] == "hover"

In [36]:
# ========= 2. Define features & target =========
target_col = "power_w"

feature_cols = ['wind_speed', 
                'wind_angle', 
                'roll', 
                'pitch', 
                'yaw', 
                'velocity_x', 
                'velocity_y', 
                'velocity_z',
                'vz_from_alt', 
                'angular_x', 
                'angular_y', 
                'angular_z', 
                'linear_acceleration_x', 
                'linear_acceleration_y', 
                'linear_acceleration_z',
                'is_cruise',
                'is_climb',
                'is_descent',
                'is_hover']

X = df_filtered[feature_cols]
y = df_filtered[target_col]


# ========= 3. Train / test split =========
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ========= 4. Train Random Forest model =========
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

In [29]:
# ========= 6. Evaluate =========
y_pred = rf.predict(X_test)
rmse = math.sqrt(mean_squared_error(y_test, y_pred))

print("Test RMSE:", rmse)

# Optional: see feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols)
print(importances.sort_values(ascending=False))


Test RMSE: 51.206299868402326
is_hover                 0.399994
linear_acceleration_z    0.214691
velocity_z               0.106339
roll                     0.033417
yaw                      0.031392
wind_speed               0.031114
angular_z                0.030040
velocity_y               0.023539
angular_x                0.021477
velocity_x               0.018982
wind_angle               0.018943
pitch                    0.017423
angular_y                0.015766
linear_acceleration_y    0.014425
linear_acceleration_x    0.013983
is_climb                 0.006543
is_descent               0.000986
is_cruise                0.000946
dtype: float64


In [ ]:
MAE  = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
MAPE = (np.abs(y_pred - y_test) /
        y_test).mean() * 100.0

print("\n===== FINAL RANDOM FOREST MODEL RESULTS =====")
print(f"MAE  = {MAE:.2f} W")
print(f"RMSE = {RMSE:.2f} W")
print(f"MAPE = {MAPE:.2f} %")

print("\n===== ERROR BY SEGMENT (NO TRANSITION) =====")
for s in ["hover", "climb", "descent", "cruise"]:
    mask_s = X_test["is_" + s]
    mae_s  = mean_absolute_error(y_test[mask_s], y_pred[mask_s])
    rmse_s = np.sqrt(mean_squared_error(y_test[mask_s], y_pred[mask_s]))
    print(f"{s:10s} | N={mask_s.sum():6d} | MAE={mae_s:8.2f} | RMSE={rmse_s:8.2f}")



===== FINAL PHYSICS MODEL RESULTS =====
MAE  = 36.25 W
RMSE = 51.21 W
MAPE = 17.06 %
